# BaristaBot with LangGraph and Gemini
A simulated cafe ordering system. Run cells in order. Live cells require a Gemini API key and may consume quota. Automated tests use a scripted model and make no API requests. Orders are simulated, not sent to a real cafe.

## 1 Install the required versions

In [ ]:
%pip install -q "langgraph==1.0.5" "langgraph-prebuilt==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

## 2 Configure secrets
Use a Kaggle secret named GOOGLE_API_KEY, a Colab secret with notebook access enabled, or an environment variable. Never paste a key into a code cell. The fallback prompts invisibly. GEMINI_MODEL can override the model if your account cannot access Gemini 2.5 Flash.

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except ImportError:
        try:
            from google.colab import userdata
            key = userdata.get("GOOGLE_API_KEY")
        except Exception:
            key = getpass("Gemini API key: ")
    if not key:
        raise ValueError("An API key is required for live cells.")
    os.environ["GOOGLE_API_KEY"] = key
    del key


## 3 State, instructions and reusable graph functions
OrderState holds message history with add_messages, a string list of drinks, and completion state. Additional flags enforce confirmation and distinguish quitting from placing an order. Tools have explicit schemas; order mutations occur in a separate node. Only incremental message updates are returned. The definitions below support all four graph stages.

In [ ]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_google_genai import ChatGoogleGenerativeAI


class OrderState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool
    confirmed: bool
    placed: bool


DRINKS = ['Espresso', 'Americano', 'Cold Brew', 'Latte', 'Cappuccino', 'Cortado',
          'Macchiato', 'Mocha', 'Flat White', 'English Breakfast Tea', 'Green Tea',
          'Earl Grey', 'Chai Latte', 'Matcha Latte', 'London Fog', 'Steamer', 'Hot Chocolate']
MODIFIERS = ['Whole', '2%', 'Oat', 'Almond', '2% Lactose Free', 'Single', 'Double',
             'Triple', 'Quadruple', 'Decaf', 'Regular', 'Hot', 'Iced', 'vanilla sweetener',
             'hazelnut sweetener', 'caramel sauce', 'chocolate sauce',
             'sugar free vanilla sweetener', 'sugar', 'extra hot', 'one pump',
             'half caff', 'extra foam', 'dirty']
WELCOME_MSG = 'Welcome to the BaristaBot cafe. Type q to quit. How may I serve you today?'
BARISTABOT_SYSINT = ('system',
    'You are BaristaBot, a friendly simulated cafe assistant. Discuss only menu products and their history. '
    'Call get_menu before recommending or adding drinks. Use exact available drink and modifier names; '
    'ask for clarification rather than inventing products. Use add_to_order, get_order and clear_order '
    'to manage the order. To modify an item, get_order, clear_order and rebuild the complete desired order. '
    'Call confirm_order when the customer is ready. Its response may request changes: apply them and '
    'confirm again. Only call place_order after explicit confirmation. Never claim the order was sent '
    'unless place_order succeeds. All kitchen actions are simulated. Prefer one tool call at a time.')


@tool
def get_menu() -> str:
    """Return the current cafe menu and available modifiers."""
    return ('MENU\nDrinks: ' + ', '.join(DRINKS) + '\nModifiers: ' + ', '.join(MODIFIERS)
            + '\nDefaults: whole milk, double espresso, regular caffeine, hot. '
            'Regular milk means Whole; sweetened means sugar. Dirty adds an espresso shot. '
            'Soy milk is out of stock. For special requests outside the listed examples, ask the cafe staff.')


@tool
def add_to_order(drink: str, modifiers: list[str]) -> str:
    """Add a menu drink with available modifiers to the order."""
    raise RuntimeError('Handled by the ordering node')


@tool
def get_order() -> str:
    """Retrieve the current order."""
    raise RuntimeError('Handled by the ordering node')


@tool
def clear_order() -> str:
    """Clear all order items before rebuilding a modified order."""
    raise RuntimeError('Handled by the ordering node')


@tool
def confirm_order() -> str:
    """Show the exact order and ask the customer to confirm or request changes."""
    raise RuntimeError('Handled by the ordering node')


@tool
def place_order() -> str:
    """Place a nonempty, confirmed order in the simulated kitchen."""
    raise RuntimeError('Handled by the ordering node')


ORDER_TOOLS = [add_to_order, get_order, clear_order, confirm_order, place_order]
MENU_NODE = ToolNode([get_menu])
QUIT = {'q', 'quit', 'exit', 'goodbye'}


def initial_state():
    return {'messages': [], 'order': [], 'finished': False, 'confirmed': False, 'placed': False}


def create_model():
    if not os.environ.get('GOOGLE_API_KEY'):
        raise ValueError('Set GOOGLE_API_KEY in notebook secrets or your environment first.')
    return ChatGoogleGenerativeAI(model=os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'), temperature=0)


def make_order_node(input_fn=input, output_fn=print):
    def order_node(state):
        order = list(state.get('order', []))
        confirmed = state.get('confirmed', False)
        finished = state.get('finished', False)
        placed = state.get('placed', False)
        results = []
        for call in state['messages'][-1].tool_calls:
            name, args = call['name'], call['args']
            if finished:
                response = 'Conversation closed; no further order changes accepted.'
            elif name == 'get_menu':
                # Mixed menu/order batches still use the real ToolNode, one call at a time.
                result = MENU_NODE.invoke({'messages': [AIMessage(content='', tool_calls=[call])]})
                results.extend(result['messages'])
                continue
            elif name == 'add_to_order':
                drink, mods = args.get('drink'), args.get('modifiers', [])
                if drink not in DRINKS or not isinstance(mods, list) or any(m not in MODIFIERS for m in mods):
                    response = 'Unavailable drink or modifier. Call get_menu and clarify with the customer.'
                else:
                    order.append(f"{drink} ({', '.join(mods) if mods else 'default modifiers'})")
                    confirmed = False
                    response = '\n'.join(order)
            elif name == 'get_order':
                response = '\n'.join(order) or '(no order)'
            elif name == 'clear_order':
                order = []
                confirmed = False
                response = 'Order cleared.'
            elif name == 'confirm_order':
                confirmed = False
                if not order:
                    response = 'Cannot confirm an empty order.'
                else:
                    output_fn('Your order:\n' + '\n'.join(order))
                    response = input_fn('Is this correct? Type yes, request changes, or q to quit: ')
                    answer = response.strip().lower()
                    confirmed = answer in {'yes', 'y', 'confirm', 'confirmed'}
                    finished = answer in QUIT
            elif name == 'place_order':
                if not order or not confirmed:
                    response = 'Not placed: a nonempty order needs explicit customer confirmation.'
                else:
                    output_fn('Sending order to simulated kitchen:\n' + '\n'.join(order))
                    finished = placed = True
                    response = 'Order placed. Estimated preparation time: 3 minutes (simulation).'
            else:
                response = f'Unknown tool: {name}. Use the provided tools.'
            results.append(ToolMessage(content=str(response), name=name, tool_call_id=call['id']))
        return {'messages': results, 'order': order, 'confirmed': confirmed,
                'finished': finished, 'placed': placed}
    return order_node


def build_graph(model, stage='ordering', input_fn=input, output_fn=print):
    """Build single-turn, human-loop, menu or complete ordering graphs."""
    if stage not in {'single', 'human', 'menu', 'ordering'}:
        raise ValueError('Unknown graph stage')
    tools = [get_menu] + (ORDER_TOOLS if stage == 'ordering' else [])
    bound = model.bind_tools(tools) if stage in {'menu', 'ordering'} else model

    def chatbot(state):
        if state.get('finished'):
            text = 'Thank you! Your order is placed. Goodbye!' if state.get('placed') else 'Goodbye! No order was placed.'
            output_fn(text)
            return {'messages': [AIMessage(content=text)]}
        if not state.get('messages'):
            return {'messages': [AIMessage(content=WELCOME_MSG)]}
        return {'messages': [bound.invoke([BARISTABOT_SYSINT] + state['messages'])]}

    def human_node(state):
        output_fn('BaristaBot: ' + str(state['messages'][-1].content))
        text = input_fn('You: ')
        return {'messages': [HumanMessage(content=text)], 'finished': text.strip().lower() in QUIT,
                'confirmed': False}

    def route(state):
        if state.get('finished'):
            return END
        calls = getattr(state['messages'][-1], 'tool_calls', [])
        if calls:
            return 'tools' if all(c['name'] == 'get_menu' for c in calls) else 'ordering'
        return 'human'

    graph = StateGraph(OrderState)
    graph.add_node('chatbot', chatbot)
    graph.add_edge(START, 'chatbot')
    if stage == 'single':
        graph.add_edge('chatbot', END)
    else:
        graph.add_node('human', human_node)
        graph.add_edge('human', 'chatbot')
        if stage == 'human':
            graph.add_conditional_edges('chatbot', lambda s: END if s.get('finished') else 'human', ['human', END])
        else:
            graph.add_node('tools', MENU_NODE)
            graph.add_edge('tools', 'chatbot')
            if stage == 'ordering':
                graph.add_node('ordering', make_order_node(input_fn, output_fn))
                graph.add_edge('ordering', 'chatbot')
                graph.add_conditional_edges('chatbot', route, ['tools', 'ordering', 'human', END])
            else:
                graph.add_conditional_edges('chatbot', route, ['tools', 'human', END])
    return graph.compile()




## 4 and 5 Single turn graph and visualization

In [ ]:
from IPython.display import Image, display
from pprint import pprint
llm = create_model()
chat_graph = build_graph(llm, "single")
def show_graph(graph):
    print(graph.get_graph().draw_mermaid())
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception as error:
        print("Image rendering unavailable; Mermaid source is shown above.", type(error).__name__)
show_graph(chat_graph)


## 6 Invoke the first turn

In [ ]:
user_msg = HumanMessage(content="Hello! Can I ask about cafe drinks?")
state = chat_graph.invoke(initial_state() | {"messages": [user_msg]})
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")


## 7 Preserve history for a second turn

In [ ]:
user_msg = HumanMessage(content="What does espresso mean?")
state["messages"].append(user_msg)
state = chat_graph.invoke(state)
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")


## 8 and 9 Human loop and conditional exit
The human node collects input. The chatbot checks finished and the conditional edge reaches END after a farewell. Type q to finish this demonstration before running the next cell.

In [ ]:
chat_with_human_graph = build_graph(llm, "human")
show_graph(chat_with_human_graph)
state = chat_with_human_graph.invoke(initial_state(), {"recursion_limit": 100})


## 10 ToolNode for the menu
get_menu reads the current menu constants on every call. This is a simulation of live stock, not an external stock service. Ask for the menu, then quit.

In [ ]:
graph_with_menu = build_graph(llm, "menu")
show_graph(graph_with_menu)
state = graph_with_menu.invoke(initial_state(), {"recursion_limit": 100})


## 11 and 12 Complete ordering graph
The ordering node handles add_to_order, get_order, clear_order, confirm_order and place_order. A modification invalidates confirmation. Mixed menu and order calls are processed in order; menu execution still uses ToolNode. Explicit yes is required for the exact displayed order. Empty and unconfirmed orders cannot be placed. The graph issues a deterministic farewell after placement so another model call cannot reopen the order.

Try an oat latte, change it to almond milk, ask for the exact order and confirm it. Try unavailable soy milk and a non-menu drink. At confirmation type a modification instead of yes, or q to cancel. Special requests use the listed examples; other requests are referred to staff.

In [ ]:
graph_with_order_tools = build_graph(llm)
show_graph(graph_with_order_tools)
state = graph_with_order_tools.invoke(initial_state(), {"recursion_limit": 100})
pprint({"order": state["order"], "finished": state["finished"], "placed": state["placed"]})


## Offline verification
These tests use actual LangGraph transitions and ToolNode with a scripted model. They do not establish live Gemini behavior. They can run without executing the secret configuration or live demonstration cells.

In [ ]:
import unittest
from langchain_core.messages import AIMessage, HumanMessage


def call(name, **args):
    return {'name': name, 'args': args, 'id': name + str(args), 'type': 'tool_call'}


class FakeModel:
    def __init__(self, responses):
        self.responses = iter(responses)
    def bind_tools(self, tools):
        return self
    def invoke(self, messages):
        return next(self.responses)


class BaristaTests(unittest.TestCase):
    def node(self, state, calls, answers=()):
        replies = iter(answers)
        node = make_order_node(lambda _: next(replies), lambda _: None)
        return node({**state, 'messages': [AIMessage(content='', tool_calls=calls)]})

    def test_guard_and_validation(self):
        state = initial_state()
        result = self.node(state, [call('place_order'), call('add_to_order', drink='Latte', modifiers=['Soy'])])
        self.assertFalse(result['placed'])
        self.assertEqual(result['order'], [])
        self.assertEqual(state['order'], [])

    def test_confirm_modify_and_reconfirm(self):
        state = self.node(initial_state(), [call('add_to_order', drink='Latte', modifiers=['Oat']), call('confirm_order')], ['yes'])
        changed = self.node(state, [call('add_to_order', drink='Green Tea', modifiers=[]), call('place_order')])
        self.assertFalse(changed['placed'])
        declined = self.node(changed, [call('confirm_order'), call('place_order')], ['change the milk'])
        self.assertFalse(declined['placed'])
        placed = self.node(declined, [call('confirm_order'), call('place_order'), call('clear_order')], ['yes'])
        self.assertTrue(placed['placed'])
        self.assertEqual(len(placed['order']), 2)
        self.assertEqual(len(placed['messages']), 3)

    def test_clear_get_quit(self):
        state = self.node(initial_state(), [call('add_to_order', drink='Espresso', modifiers=[]), call('get_order')])
        self.assertIn('Espresso', state['messages'][-1].content)
        cleared = self.node(state, [call('clear_order'), call('confirm_order')])
        self.assertEqual(cleared['order'], [])
        quit_state = self.node(state, [call('confirm_order'), call('place_order')], [' Q '])
        self.assertTrue(quit_state['finished'])
        self.assertFalse(quit_state['placed'])

    def test_full_graph_mixed_tools(self):
        model = FakeModel([
            AIMessage(content='', tool_calls=[call('get_menu'), call('add_to_order', drink='Latte', modifiers=['Oat'])]),
            AIMessage(content='', tool_calls=[call('confirm_order')]),
            AIMessage(content='', tool_calls=[call('place_order')]),
        ])
        replies = iter(['One oat latte', 'yes'])
        graph = build_graph(model, input_fn=lambda _: next(replies), output_fn=lambda _: None)
        result = graph.invoke(initial_state(), {'recursion_limit': 100})
        self.assertTrue(result['placed'])
        self.assertIn('Goodbye', result['messages'][-1].content)
        tool_ids = [m.tool_call_id for m in result['messages'] if m.type == 'tool']
        self.assertEqual(len(tool_ids), 4)
        self.assertEqual(len(result['order']), 1)

    def test_single_turn_and_second_turn(self):
        graph = build_graph(FakeModel([AIMessage(content='Hello'), AIMessage(content='Menu questions welcome')]), 'single')
        state = initial_state() | {'messages': [HumanMessage(content='Hi')]}
        state = graph.invoke(state)
        self.assertEqual(len(state['messages']), 2)
        state['messages'].append(HumanMessage(content='Can I ask about coffee?'))
        self.assertEqual(len(graph.invoke(state)['messages']), 4)

    def test_menu_toolnode_and_human_exit(self):
        replies = iter(['menu', 'quit'])
        model = FakeModel([AIMessage(content='', tool_calls=[call('get_menu')]), AIMessage(content='Here is the menu')])
        result = build_graph(model, 'menu', lambda _: next(replies), lambda _: None).invoke(initial_state())
        self.assertTrue(result['finished'])
        self.assertTrue(any(m.type == 'tool' and 'Soy milk' in m.content for m in result['messages']))
        result = build_graph(FakeModel([]), 'human', lambda _: 'EXIT', lambda _: None).invoke(initial_state())
        self.assertTrue(result['finished'])



suite = unittest.defaultTestLoader.loadTestsFromTestCase(BaristaTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()


## Notes and sources
- The stable model ID is gemini-2.5-flash, rather than the lesson’s gemini-2.5-flash-latest. Model access depends on your account; set GEMINI_MODEL to an available tool-capable model if necessary.
- API authentication, quota and network errors surface to the caller; no order is silently assumed successful.
- A recursion limit of 100 bounds loops; reaching it stops execution instead of placing an order.
- [Gemini model documentation](https://ai.google.dev/gemini-api/docs/models/gemini-2.5-flash)
- [LangGraph ToolNode](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode)
- No live API result is included: tests here are offline. Restart a demo with initial_state() for a new conversation.